In [1]:
# Variaveis de sessão e organização de credenciais de usuário senhas e sessões.

spark.conf.set(
    "spark.sql.catalog.aicat.s3.access-key-id",
    "9340b73b9fa4150db3d6d85238faf7912a8fac1f"
)

spark.conf.set(
    "spark.sql.catalog.aicat.s3.secret-access-key",
    aidputils.secrets.get(name="SECRET_KEY_OBS", key="SECRET_KEY")
)
 
spark.conf.set(
    "spark.sql.catalog.aicat.credential",
    aidputils.secrets.get(name="ICEBERG_REST_CATALOG", key="USER:PASSWORD")
)

spark.conf.set(
    "spark.sql.catalog.aicat.scope",
    "PRINCIPAL_ROLE:ALL"
)


# Nome lógico do catálogo no Spark
spark.conf.set(
    "spark.sql.catalog.aicat",
    "org.apache.iceberg.spark.SparkCatalog"
)

# Tipo de catálogo
spark.conf.set(
    "spark.sql.catalog.aicat.type",
    "rest"
)

# Endpoint REST do Oracle AI Data Catalog
spark.conf.set(
    "spark.sql.catalog.aicat.uri",
    "https://G84191C2534E979-POCLAKEHOUSEICEBERG.adb.sa-saopaulo-1.oraclecloudapps.com/catalog"
)

# Nome do warehouse do catálogo Iceberg
spark.conf.set(
    "spark.sql.catalog.aicat.warehouse",
    "aicat_iceberg_rest_catalog"
)

# Autenticação OAuth 2.0
spark.conf.set(
    "spark.sql.catalog.aicat.rest.auth.type",
    "oauth2"
)

spark.conf.set(
    "spark.sql.catalog.aicat.oauth2-server-uri",
    "https://G84191C2534E979-POCLAKEHOUSEICEBERG.adb.sa-saopaulo-1.oraclecloudapps.com/catalog/v1/auth/token"
)

print("Configuração do catálogo REST aplicada.")

Configuração do catálogo REST aplicada.


In [2]:
CAT = "aicat"

BRONZE = f"{CAT}.BRONZE.CENSO_ESCOLAR_2024"
PRATA = f"{CAT}.PRATA.DP01_ESTABELECIMENTOS_ENSINO"

df_bronze = spark.table(BRONZE)

print("Tabela de origem:", BRONZE)
print("Registros:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))

df_bronze.select(
    "NU_ANO_CENSO",
    "CO_ENTIDADE",
    "NO_ENTIDADE",
    "SG_UF",
    "NO_MUNICIPIO",
    "TP_DEPENDENCIA",
    "TP_LOCALIZACAO",
    "TP_SITUACAO_FUNCIONAMENTO"
).show(10, truncate=False)

Tabela de origem: aicat.BRONZE.CENSO_ESCOLAR_2024


Registros: 215545
Colunas: 426


+------------+-----------+------------------------------------------+-----+-----------------+--------------+--------------+-------------------------+
|NU_ANO_CENSO|CO_ENTIDADE|NO_ENTIDADE                               |SG_UF|NO_MUNICIPIO     |TP_DEPENDENCIA|TP_LOCALIZACAO|TP_SITUACAO_FUNCIONAMENTO|
+------------+-----------+------------------------------------------+-----+-----------------+--------------+--------------+-------------------------+
|2024        |24027065   |ESC ISOLADA ALZIRA SORIANO                |RN   |Jardim de Angicos|2             |2             |2                        |
|2024        |24027081   |ESC ISOLADA SERRINHA                      |RN   |Jardim de Angicos|2             |2             |2                        |
|2024        |24027090   |ESC MUL CEL JOSE SOARES BILRO             |RN   |Jardim de Angicos|3             |2             |2                        |
|2024        |24027138   |ESC MUL JOSE AUGUSTO                      |RN   |Jardim de Angicos|3      

In [10]:
colunas_categoricas = [
    "TP_DEPENDENCIA",
    "TP_CATEGORIA_ESCOLA_PRIVADA",
    "TP_LOCALIZACAO",
    "TP_LOCALIZACAO_DIFERENCIADA",
    "TP_SITUACAO_FUNCIONAMENTO",
    "TP_REGULAMENTACAO",
    "TP_RESPONSAVEL_REGULAMENTACAO",
    "TP_REDE_LOCAL"
]

for coluna in colunas_categoricas:
    print(f"\n--- {coluna} ---")

    (
        df_bronze
        .groupBy(coluna)
        .count()
        .orderBy(coluna)
        .show(30, truncate=False)
    )


--- TP_DEPENDENCIA ---


+--------------+------+
|TP_DEPENDENCIA|count |
+--------------+------+
|1             |724   |
|2             |33254 |
|3             |128999|
|4             |52568 |
+--------------+------+


--- TP_CATEGORIA_ESCOLA_PRIVADA ---


+---------------------------+------+
|TP_CATEGORIA_ESCOLA_PRIVADA|count |
+---------------------------+------+
|NULL                       |172327|
|1                          |31953 |
|2                          |2299  |
|3                          |467   |
|4                          |8499  |
+---------------------------+------+


--- TP_LOCALIZACAO ---


+--------------+------+
|TP_LOCALIZACAO|count |
+--------------+------+
|1             |144456|
|2             |71089 |
+--------------+------+


--- TP_LOCALIZACAO_DIFERENCIADA ---


+---------------------------+------+
|TP_LOCALIZACAO_DIFERENCIADA|count |
+---------------------------+------+
|NULL                       |34480 |
|0                          |169339|
|1                          |4480  |
|2                          |3558  |
|3                          |2619  |
|8                          |1069  |
+---------------------------+------+


--- TP_SITUACAO_FUNCIONAMENTO ---


+-------------------------+------+
|TP_SITUACAO_FUNCIONAMENTO|count |
+-------------------------+------+
|1                        |181065|
|2                        |31321 |
|3                        |3159  |
+-------------------------+------+


--- TP_REGULAMENTACAO ---


+-----------------+------+
|TP_REGULAMENTACAO|count |
+-----------------+------+
|NULL             |34480 |
|0                |4618  |
|1                |161974|
|2                |14473 |
+-----------------+------+


--- TP_RESPONSAVEL_REGULAMENTACAO ---


+-----------------------------+------+
|TP_RESPONSAVEL_REGULAMENTACAO|count |
+-----------------------------+------+
|NULL                         |39098 |
|1                            |1274  |
|2                            |67602 |
|3                            |101544|
|4                            |5770  |
|5                            |257   |
+-----------------------------+------+


--- TP_REDE_LOCAL ---


+-------------+-----+
|TP_REDE_LOCAL|count|
+-------------+-----+
|NULL         |34480|
|0            |13266|
|1            |32358|
|2            |34258|
|3            |75979|
|9            |25204|
+-------------+-----+



In [13]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

duplicidades = (
    df_bronze
    .groupBy("NU_ANO_CENSO", "CO_ENTIDADE")
    .count()
    .filter(F.col("count") > 1)
)

print("Quantidade de chaves duplicadas:", duplicidades.count())

duplicidades.show(20, truncate=False)

Quantidade de chaves duplicadas: 0


+------------+-----------+-----+
|NU_ANO_CENSO|CO_ENTIDADE|count|
+------------+-----------+-----+
+------------+-----------+-----+



In [14]:
colunas_dp01 = [
    # Identificação
    "NU_ANO_CENSO",
    "CO_ENTIDADE",
    "NO_ENTIDADE",

    # Localização
    "NO_REGIAO",
    "CO_REGIAO",
    "NO_UF",
    "SG_UF",
    "CO_UF",
    "NO_MUNICIPIO",
    "CO_MUNICIPIO",
    "NO_DISTRITO",
    "CO_DISTRITO",
    "DS_ENDERECO",
    "NU_ENDERECO",
    "DS_COMPLEMENTO",
    "NO_BAIRRO",
    "CO_CEP",

    # Gestão
    "TP_DEPENDENCIA",
    "TP_CATEGORIA_ESCOLA_PRIVADA",
    "TP_LOCALIZACAO",
    "TP_LOCALIZACAO_DIFERENCIADA",

    # Funcionamento
    "TP_SITUACAO_FUNCIONAMENTO",
    "TP_REGULAMENTACAO",
    "TP_RESPONSAVEL_REGULAMENTACAO",

    # Infraestrutura básica
    "IN_AGUA_POTAVEL",
    "IN_AGUA_REDE_PUBLICA",
    "IN_AGUA_POCO_ARTESIANO",
    "IN_AGUA_CACIMBA",
    "IN_AGUA_FONTE_RIO",
    "IN_AGUA_INEXISTENTE",
    "IN_AGUA_CARRO_PIPA",
    "IN_ENERGIA_REDE_PUBLICA",
    "IN_ENERGIA_GERADOR_FOSSIL",
    "IN_ENERGIA_RENOVAVEL",
    "IN_ENERGIA_INEXISTENTE",
    "IN_ESGOTO_REDE_PUBLICA",
    "IN_ESGOTO_FOSSA_SEPTICA",
    "IN_ESGOTO_FOSSA_COMUM",
    "IN_ESGOTO_FOSSA",
    "IN_ESGOTO_INEXISTENTE",
    "IN_LIXO_SERVICO_COLETA",
    "IN_LIXO_QUEIMA",
    "IN_LIXO_ENTERRA",
    "IN_LIXO_DESTINO_FINAL_PUBLICO",
    "IN_LIXO_DESCARTA_OUTRA_AREA"
]

df_dp01 = df_bronze.select(*colunas_dp01)

print("Registros:", df_dp01.count())
print("Colunas selecionadas:", len(df_dp01.columns))

df_dp01.printSchema()

Registros: 215545
Colunas selecionadas: 45
root
 |-- NU_ANO_CENSO: string (nullable = true)
 |-- CO_ENTIDADE: string (nullable = true)
 |-- NO_ENTIDADE: string (nullable = true)
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_REGIAO: string (nullable = true)
 |-- NO_UF: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_UF: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_DISTRITO: string (nullable = true)
 |-- CO_DISTRITO: string (nullable = true)
 |-- DS_ENDERECO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- DS_COMPLEMENTO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- CO_CEP: string (nullable = true)
 |-- TP_DEPENDENCIA: string (nullable = true)
 |-- TP_CATEGORIA_ESCOLA_PRIVADA: string (nullable = true)
 |-- TP_LOCALIZACAO: string (nullable = true)
 |-- TP_LOCALIZACAO_DIFERENCIADA: string (nullable = true)
 |-- TP_SITUACAO_FUNCIONAMENTO: st

In [15]:
# Campos numéricos categóricos
campos_smallint = [
    "NU_ANO_CENSO",
    "TP_DEPENDENCIA",
    "TP_CATEGORIA_ESCOLA_PRIVADA",
    "TP_LOCALIZACAO",
    "TP_LOCALIZACAO_DIFERENCIADA",
    "TP_SITUACAO_FUNCIONAMENTO",
    "TP_REGULAMENTACAO",
    "TP_RESPONSAVEL_REGULAMENTACAO"
]

# Conversão dos campos
df_dp01_tipado = df_dp01

for coluna in campos_smallint:
    df_dp01_tipado = df_dp01_tipado.withColumn(
        coluna,
        F.col(coluna).cast("SMALLINT")
    )

print("Schema após tipagem:")
df_dp01_tipado.printSchema()

Schema após tipagem:
root
 |-- NU_ANO_CENSO: short (nullable = true)
 |-- CO_ENTIDADE: string (nullable = true)
 |-- NO_ENTIDADE: string (nullable = true)
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_REGIAO: string (nullable = true)
 |-- NO_UF: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_UF: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_DISTRITO: string (nullable = true)
 |-- CO_DISTRITO: string (nullable = true)
 |-- DS_ENDERECO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- DS_COMPLEMENTO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- CO_CEP: string (nullable = true)
 |-- TP_DEPENDENCIA: short (nullable = true)
 |-- TP_CATEGORIA_ESCOLA_PRIVADA: short (nullable = true)
 |-- TP_LOCALIZACAO: short (nullable = true)
 |-- TP_LOCALIZACAO_DIFERENCIADA: short (nullable = true)
 |-- TP_SITUACAO_FUNCIONAMENTO: short (nullable = true)
 |-- 

In [16]:
for coluna in campos_smallint:
    nulos_origem = df_dp01.filter(
        F.col(coluna).isNull()
    ).count()

    nulos_destino = df_dp01_tipado.filter(
        F.col(coluna).isNull()
    ).count()

    print(
        f"{coluna}: "
        f"nulos origem={nulos_origem}, "
        f"nulos após conversão={nulos_destino}"
    )

NU_ANO_CENSO: nulos origem=0, nulos após conversão=0


TP_DEPENDENCIA: nulos origem=0, nulos após conversão=0


TP_CATEGORIA_ESCOLA_PRIVADA: nulos origem=172327, nulos após conversão=172327


TP_LOCALIZACAO: nulos origem=0, nulos após conversão=0


TP_LOCALIZACAO_DIFERENCIADA: nulos origem=34480, nulos após conversão=34480


TP_SITUACAO_FUNCIONAMENTO: nulos origem=0, nulos após conversão=0


TP_REGULAMENTACAO: nulos origem=34480, nulos após conversão=34480


TP_RESPONSAVEL_REGULAMENTACAO: nulos origem=39098, nulos após conversão=39098


In [17]:
colunas_infraestrutura = [
    # Instalações
    "IN_ALMOXARIFADO",
    "IN_AREA_VERDE",
    "IN_AUDITORIO",
    "IN_BANHEIRO",
    "IN_BANHEIRO_PNE",
    "IN_BIBLIOTECA",
    "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_COZINHA",
    "IN_LABORATORIO_CIENCIAS",
    "IN_LABORATORIO_INFORMATICA",
    "IN_PATIO_COBERTO",
    "IN_PATIO_DESCOBERTO",
    "IN_PARQUE_INFANTIL",
    "IN_QUADRA_ESPORTES",
    "IN_QUADRA_ESPORTES_COBERTA",
    "IN_REFEITORIO",

    # Acessibilidade
    "IN_ACESSIBILIDADE_CORRIMAO",
    "IN_ACESSIBILIDADE_ELEVADOR",
    "IN_ACESSIBILIDADE_PISOS_TATEIS",
    "IN_ACESSIBILIDADE_RAMPAS",
    "IN_ACESSIBILIDADE_SINAL_SONORO",
    "IN_ACESSIBILIDADE_SINAL_TATIL",
    "IN_ACESSIBILIDADE_SINAL_VISUAL",
    "IN_ACESSIBILIDADE_INEXISTENTE",

    # Tecnologia e conectividade
    "IN_COMPUTADOR",
    "IN_EQUIP_IMPRESSORA",
    "IN_EQUIP_IMPRESSORA_MULT",
    "IN_EQUIP_SCANNER",
    "IN_EQUIP_TV",
    "IN_EQUIP_LOUSA_DIGITAL",
    "IN_DESKTOP_ALUNO",
    "IN_COMP_PORTATIL_ALUNO",
    "IN_TABLET_ALUNO",
    "IN_INTERNET",
    "IN_INTERNET_ALUNOS",
    "IN_INTERNET_ADMINISTRATIVO",
    "IN_INTERNET_APRENDIZAGEM",
    "IN_ACESSO_INTERNET_COMPUTADOR",
    "IN_ACES_INTERNET_DISP_PESSOAIS",
    "IN_BANDA_LARGA"
]

colunas_dp01_completas = colunas_dp01 + colunas_infraestrutura

df_dp01 = df_bronze.select(*colunas_dp01_completas)

print("Registros:", df_dp01.count())
print("Colunas:", len(df_dp01.columns))

Registros: 215545
Colunas: 85


In [18]:
valores_indicadores = df_dp01.agg(
    *[
        F.collect_set(F.col(c)).alias(c)
        for c in colunas_infraestrutura
    ]
).first().asDict()

for coluna, valores in valores_indicadores.items():
    print(f"{coluna}: {sorted(valores or [])}")

IN_ALMOXARIFADO: ["0", "1"]
IN_AREA_VERDE: ["0", "1"]
IN_AUDITORIO: ["0", "1"]
IN_BANHEIRO: ["0", "1"]
IN_BANHEIRO_PNE: ["0", "1"]
IN_BIBLIOTECA: ["0", "1"]
IN_BIBLIOTECA_SALA_LEITURA: ["0", "1"]
IN_COZINHA: ["0", "1"]
IN_LABORATORIO_CIENCIAS: ["0", "1"]
IN_LABORATORIO_INFORMATICA: ["0", "1"]
IN_PATIO_COBERTO: ["0", "1"]
IN_PATIO_DESCOBERTO: ["0", "1"]
IN_PARQUE_INFANTIL: ["0", "1"]
IN_QUADRA_ESPORTES: ["0", "1"]
IN_QUADRA_ESPORTES_COBERTA: ["0", "1"]
IN_REFEITORIO: ["0", "1"]
IN_ACESSIBILIDADE_CORRIMAO: ["0", "1"]
IN_ACESSIBILIDADE_ELEVADOR: ["0", "1"]
IN_ACESSIBILIDADE_PISOS_TATEIS: ["0", "1"]
IN_ACESSIBILIDADE_RAMPAS: ["0", "1"]
IN_ACESSIBILIDADE_SINAL_SONORO: ["0", "1"]
IN_ACESSIBILIDADE_SINAL_TATIL: ["0", "1"]
IN_ACESSIBILIDADE_SINAL_VISUAL: ["0", "1"]
IN_ACESSIBILIDADE_INEXISTENTE: ["0", "1"]
IN_COMPUTADOR: ["0", "1"]
IN_EQUIP_IMPRESSORA: ["0", "1"]
IN_EQUIP_IMPRESSORA_MULT: ["0", "1"]
IN_EQUIP_SCANNER: ["0", "1"]
IN_EQUIP_TV: ["0", "1"]
IN_EQUIP_LOUSA_DIGITAL: ["0", "1"]
IN_DESK

In [19]:
# DataFrame tipado
df_dp01_tipado = df_dp01

for coluna in colunas_infraestrutura:
    df_dp01_tipado = df_dp01_tipado.withColumn(
        coluna,
        F.col(coluna).cast("SMALLINT")
    )

print("Registros:", df_dp01_tipado.count())
print("Colunas:", len(df_dp01_tipado.columns))

df_dp01_tipado.printSchema()

Registros: 215545
Colunas: 85
root
 |-- NU_ANO_CENSO: string (nullable = true)
 |-- CO_ENTIDADE: string (nullable = true)
 |-- NO_ENTIDADE: string (nullable = true)
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_REGIAO: string (nullable = true)
 |-- NO_UF: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_UF: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_DISTRITO: string (nullable = true)
 |-- CO_DISTRITO: string (nullable = true)
 |-- DS_ENDERECO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- DS_COMPLEMENTO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- CO_CEP: string (nullable = true)
 |-- TP_DEPENDENCIA: string (nullable = true)
 |-- TP_CATEGORIA_ESCOLA_PRIVADA: string (nullable = true)
 |-- TP_LOCALIZACAO: string (nullable = true)
 |-- TP_LOCALIZACAO_DIFERENCIADA: string (nullable = true)
 |-- TP_SITUACAO_FUNCIONAMENTO: string (nullabl

In [20]:
for coluna in colunas_infraestrutura:
    valores = [
        row[coluna]
        for row in (
            df_dp01_tipado
            .select(coluna)
            .distinct()
            .collect()
        )
    ]

    print(f"{coluna}: {sorted(valores, key=lambda x: -1 if x is None else x)}")

IN_ALMOXARIFADO: [None, 0, 1]


IN_AREA_VERDE: [None, 0, 1]


IN_AUDITORIO: [None, 0, 1]


IN_BANHEIRO: [None, 0, 1]


IN_BANHEIRO_PNE: [None, 0, 1]


IN_BIBLIOTECA: [None, 0, 1]


... (30 outputs left out)

IN_INTERNET_APRENDIZAGEM: [None, 0, 1]


IN_ACESSO_INTERNET_COMPUTADOR: [None, 0, 1, 9]


IN_ACES_INTERNET_DISP_PESSOAIS: [None, 0, 1, 9]


IN_BANDA_LARGA: [None, 0, 1]


In [21]:
for coluna in colunas_infraestrutura:
    resultado = (
        df_dp01_tipado
        .groupBy(coluna)
        .count()
        .orderBy(coluna)
    )

    print(f"\n--- {coluna} ---")
    resultado.show(truncate=False)


--- IN_ALMOXARIFADO ---


+---------------+-----+
|IN_ALMOXARIFADO|count|
+---------------+-----+
|NULL           |34480|
|0              |86306|
|1              |94759|
+---------------+-----+


--- IN_AREA_VERDE ---


+-------------+------+
|IN_AREA_VERDE|count |
+-------------+------+
|NULL         |34480 |
|0            |117660|
|1            |63405 |
+-------------+------+


--- IN_AUDITORIO ---


+------------+------+
|IN_AUDITORIO|count |
+------------+------+
|NULL        |34480 |
|0           |157894|
|1           |23171 |
+------------+------+


--- IN_BANHEIRO ---


+-----------+------+
|IN_BANHEIRO|count |
+-----------+------+
|NULL       |34480 |
|0          |4925  |
|1          |176140|
+-----------+------+


--- IN_BANHEIRO_PNE ---


+---------------+-----+
|IN_BANHEIRO_PNE|count|
+---------------+-----+
|NULL           |34480|
|0              |81700|
|1              |99365|
+---------------+-----+


--- IN_BIBLIOTECA ---


... (31 outputs left out)

+------------------------+------+
|IN_INTERNET_APRENDIZAGEM|count |
+------------------------+------+
|NULL                    |34480 |
|0                       |60145 |
|1                       |120920|
+------------------------+------+


--- IN_ACESSO_INTERNET_COMPUTADOR ---


+-----------------------------+------+
|IN_ACESSO_INTERNET_COMPUTADOR|count |
+-----------------------------+------+
|NULL                         |34480 |
|0                            |119712|
|1                            |57095 |
|9                            |4258  |
+-----------------------------+------+


--- IN_ACES_INTERNET_DISP_PESSOAIS ---


+------------------------------+------+
|IN_ACES_INTERNET_DISP_PESSOAIS|count |
+------------------------------+------+
|NULL                          |34480 |
|0                             |141453|
|1                             |35354 |
|9                             |4258  |
+------------------------------+------+


--- IN_BANDA_LARGA ---


+--------------+------+
|IN_BANDA_LARGA|count |
+--------------+------+
|NULL          |48774 |
|0             |21576 |
|1             |145195|
+--------------+------+



In [22]:
# DataFrame definitivo do DP01
df_dp01_final = df_dp01_tipado

print("Total de registros:", df_dp01_final.count())
print("Total de colunas:", len(df_dp01_final.columns))

# Conferir chave de negócio
print(
    "Chaves distintas:",
    df_dp01_final
    .select("NU_ANO_CENSO", "CO_ENTIDADE")
    .distinct()
    .count()
)

# Conferir amostra
df_dp01_final.show(5, truncate=False)

Total de registros: 215545
Total de colunas: 85


Chaves distintas: 215545


+------------+-----------+-------------------------------+---------+---------+-------------------+-----+-----+-----------------+------------+-----------------+-----------+----------------+-----------+--------------+----------+--------+--------------+---------------------------+--------------+---------------------------+-------------------------+-----------------+-----------------------------+---------------+--------------------+----------------------+---------------+-----------------+-------------------+------------------+-----------------------+-------------------------+--------------------+----------------------+----------------------+-----------------------+---------------------+---------------+---------------------+----------------------+--------------+---------------+-----------------------------+---------------------------+---------------+-------------+------------+-----------+---------------+-------------+--------------------------+----------+-----------------------+------------

In [25]:
# Reconstruir o DataFrame a partir da seleção original
df_dp01_tipado = df_dp01

# Converter campos categóricos para SMALLINT
for coluna in campos_smallint:
    df_dp01_tipado = df_dp01_tipado.withColumn(
        coluna, F.col(coluna).cast("smallint")
    )

# Converter indicadores para SMALLINT
for coluna in colunas_infraestrutura:
    df_dp01_tipado = df_dp01_tipado.withColumn(
        coluna, F.col(coluna).cast("smallint")
    )

df_dp01_final = df_dp01_tipado

df_dp01_final.printSchema()

root
 |-- NU_ANO_CENSO: short (nullable = true)
 |-- CO_ENTIDADE: string (nullable = true)
 |-- NO_ENTIDADE: string (nullable = true)
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_REGIAO: string (nullable = true)
 |-- NO_UF: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_UF: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_DISTRITO: string (nullable = true)
 |-- CO_DISTRITO: string (nullable = true)
 |-- DS_ENDERECO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- DS_COMPLEMENTO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- CO_CEP: string (nullable = true)
 |-- TP_DEPENDENCIA: short (nullable = true)
 |-- TP_CATEGORIA_ESCOLA_PRIVADA: short (nullable = true)
 |-- TP_LOCALIZACAO: short (nullable = true)
 |-- TP_LOCALIZACAO_DIFERENCIADA: short (nullable = true)
 |-- TP_SITUACAO_FUNCIONAMENTO: short (nullable = true)
 |-- TP_REGULAMENTACAO: sh

In [26]:
colunas_saneamento = [
    "IN_AGUA_POTAVEL",
    "IN_AGUA_REDE_PUBLICA",
    "IN_AGUA_POCO_ARTESIANO",
    "IN_AGUA_CACIMBA",
    "IN_AGUA_FONTE_RIO",
    "IN_AGUA_INEXISTENTE",
    "IN_AGUA_CARRO_PIPA",
    "IN_ENERGIA_REDE_PUBLICA",
    "IN_ENERGIA_GERADOR_FOSSIL",
    "IN_ENERGIA_RENOVAVEL",
    "IN_ENERGIA_INEXISTENTE",
    "IN_ESGOTO_REDE_PUBLICA",
    "IN_ESGOTO_FOSSA_SEPTICA",
    "IN_ESGOTO_FOSSA_COMUM",
    "IN_ESGOTO_FOSSA",
    "IN_ESGOTO_INEXISTENTE",
    "IN_LIXO_SERVICO_COLETA",
    "IN_LIXO_QUEIMA",
    "IN_LIXO_ENTERRA",
    "IN_LIXO_DESTINO_FINAL_PUBLICO",
    "IN_LIXO_DESCARTA_OUTRA_AREA"
]

for coluna in colunas_saneamento:
    df_dp01_final = df_dp01_final.withColumn(
        coluna, F.col(coluna).cast("smallint")
    )

df_dp01_final.printSchema()

root
 |-- NU_ANO_CENSO: short (nullable = true)
 |-- CO_ENTIDADE: string (nullable = true)
 |-- NO_ENTIDADE: string (nullable = true)
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_REGIAO: string (nullable = true)
 |-- NO_UF: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_UF: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_DISTRITO: string (nullable = true)
 |-- CO_DISTRITO: string (nullable = true)
 |-- DS_ENDERECO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- DS_COMPLEMENTO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- CO_CEP: string (nullable = true)
 |-- TP_DEPENDENCIA: short (nullable = true)
 |-- TP_CATEGORIA_ESCOLA_PRIVADA: short (nullable = true)
 |-- TP_LOCALIZACAO: short (nullable = true)
 |-- TP_LOCALIZACAO_DIFERENCIADA: short (nullable = true)
 |-- TP_SITUACAO_FUNCIONAMENTO: short (nullable = true)
 |-- TP_REGULAMENTACAO: sh

In [33]:
# Converter todos os campos numéricos para INT
colunas_numericas = campos_smallint + colunas_infraestrutura + colunas_saneamento

for coluna in colunas_numericas:
    df_dp01_final = df_dp01_final.withColumn(
        coluna, F.col(coluna).cast("int")
    )

df_dp01_final.printSchema()

root
 |-- NU_ANO_CENSO: integer (nullable = true)
 |-- CO_ENTIDADE: string (nullable = true)
 |-- NO_ENTIDADE: string (nullable = true)
 |-- NO_REGIAO: string (nullable = true)
 |-- CO_REGIAO: string (nullable = true)
 |-- NO_UF: string (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_UF: string (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- CO_MUNICIPIO: string (nullable = true)
 |-- NO_DISTRITO: string (nullable = true)
 |-- CO_DISTRITO: string (nullable = true)
 |-- DS_ENDERECO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- DS_COMPLEMENTO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- CO_CEP: string (nullable = true)
 |-- TP_DEPENDENCIA: integer (nullable = true)
 |-- TP_CATEGORIA_ESCOLA_PRIVADA: integer (nullable = true)
 |-- TP_LOCALIZACAO: integer (nullable = true)
 |-- TP_LOCALIZACAO_DIFERENCIADA: integer (nullable = true)
 |-- TP_SITUACAO_FUNCIONAMENTO: integer (nullable = true)
 |-- TP_REGULA

In [34]:
colunas_sql = ",\n".join(
    f"`{campo.name}` {campo.dataType.simpleString().upper()}"
    for campo in df_dp01_final.schema.fields
)

ddl = f"""
CREATE TABLE aicat.PRATA.DP01_ESTABELECIMENTOS_ENSINO
(
    {colunas_sql}
)
USING iceberg
"""

spark.sql(ddl)

print("Tabela recriada com o esquema definitivo!")

Tabela recriada com o esquema definitivo!


In [35]:
spark.sql("""
DESCRIBE TABLE aicat.PRATA.DP01_ESTABELECIMENTOS_ENSINO
""").show(100, truncate=False)

+------------------------------+---------+-------+
|col_name                      |data_type|comment|
+------------------------------+---------+-------+
|NU_ANO_CENSO                  |int      |NULL   |
|CO_ENTIDADE                   |string   |NULL   |
|NO_ENTIDADE                   |string   |NULL   |
|NO_REGIAO                     |string   |NULL   |
|CO_REGIAO                     |string   |NULL   |
|NO_UF                         |string   |NULL   |
|SG_UF                         |string   |NULL   |
|CO_UF                         |string   |NULL   |
|NO_MUNICIPIO                  |string   |NULL   |
|CO_MUNICIPIO                  |string   |NULL   |
|NO_DISTRITO                   |string   |NULL   |
|CO_DISTRITO                   |string   |NULL   |
|DS_ENDERECO                   |string   |NULL   |
|NU_ENDERECO                   |string   |NULL   |
|DS_COMPLEMENTO                |string   |NULL   |
|NO_BAIRRO                     |string   |NULL   |
|CO_CEP                        

In [36]:
df_dp01_final.createOrReplaceTempView(
    "vw_dp01_estabelecimentos_ensino"
)

spark.sql("""
    INSERT OVERWRITE TABLE aicat.PRATA.DP01_ESTABELECIMENTOS_ENSINO
    SELECT *
    FROM vw_dp01_estabelecimentos_ensino
""")

print("Carga do DP01 concluída!")

Carga do DP01 concluída!


In [37]:
df_prata = spark.table(
    "aicat.PRATA.DP01_ESTABELECIMENTOS_ENSINO"
)

print("Total de registros:", df_prata.count())
print("Total de colunas:", len(df_prata.columns))

print(
    "Chaves distintas:",
    df_prata
    .select("NU_ANO_CENSO", "CO_ENTIDADE")
    .distinct()
    .count()
)

Total de registros: 215545
Total de colunas: 85


Chaves distintas: 215545


In [38]:
campos_obrigatorios = [
    "NU_ANO_CENSO",
    "CO_ENTIDADE",
    "NO_ENTIDADE",
    "CO_UF",
    "SG_UF",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO"
]

for coluna in campos_obrigatorios:
    qtd_nulos = df_prata.filter(
        F.col(coluna).isNull() |
        (F.trim(F.col(coluna)) == "")
    ).count()

    print(f"{coluna}: {qtd_nulos}")

NU_ANO_CENSO: 0


CO_ENTIDADE: 0


NO_ENTIDADE: 0


CO_UF: 0


SG_UF: 0


CO_MUNICIPIO: 0


NO_MUNICIPIO: 0


In [39]:
campos_endereco = [
    "DS_ENDERECO",
    "NU_ENDERECO",
    "DS_COMPLEMENTO",
    "NO_BAIRRO",
    "CO_CEP"
]

for coluna in campos_endereco:
    qtd_nulos = df_prata.filter(
        F.col(coluna).isNull() |
        (F.trim(F.col(coluna)) == "")
    ).count()

    percentual = qtd_nulos / df_prata.count() * 100

    print(
        f"{coluna}: {qtd_nulos} "
        f"({percentual:.2f}%)"
    )

DS_ENDERECO: 1 (0.00%)


NU_ENDERECO: 41917 (19.45%)


DS_COMPLEMENTO: 144569 (67.07%)


NO_BAIRRO: 28640 (13.29%)


CO_CEP: 0 (0.00%)


In [40]:
for coluna in colunas_saneamento + colunas_infraestrutura:
    print(f"\n{coluna}")

    (
        df_prata
        .groupBy(coluna)
        .count()
        .orderBy(coluna)
        .show()
    )


IN_AGUA_POTAVEL


+---------------+------+
|IN_AGUA_POTAVEL| count|
+---------------+------+
|           NULL| 34480|
|              0|  6658|
|              1|174407|
+---------------+------+


IN_AGUA_REDE_PUBLICA


+--------------------+------+
|IN_AGUA_REDE_PUBLICA| count|
+--------------------+------+
|                NULL| 34480|
|                   0| 40089|
|                   1|140976|
+--------------------+------+


IN_AGUA_POCO_ARTESIANO


+----------------------+------+
|IN_AGUA_POCO_ARTESIANO| count|
+----------------------+------+
|                  NULL| 34480|
|                     0|151839|
|                     1| 29226|
+----------------------+------+


IN_AGUA_CACIMBA


+---------------+------+
|IN_AGUA_CACIMBA| count|
+---------------+------+
|           NULL| 34480|
|              0|166659|
|              1| 14406|
+---------------+------+


IN_AGUA_FONTE_RIO


+-----------------+------+
|IN_AGUA_FONTE_RIO| count|
+-----------------+------+
|             NULL| 34480|
|                0|174288|
|                1|  6777|
+-----------------+------+


IN_AGUA_INEXISTENTE


... (52 outputs left out)

+------------------------+------+
|IN_INTERNET_APRENDIZAGEM| count|
+------------------------+------+
|                    NULL| 34480|
|                       0| 60145|
|                       1|120920|
+------------------------+------+


IN_ACESSO_INTERNET_COMPUTADOR


+-----------------------------+------+
|IN_ACESSO_INTERNET_COMPUTADOR| count|
+-----------------------------+------+
|                         NULL| 34480|
|                            0|119712|
|                            1| 57095|
|                            9|  4258|
+-----------------------------+------+


IN_ACES_INTERNET_DISP_PESSOAIS


+------------------------------+------+
|IN_ACES_INTERNET_DISP_PESSOAIS| count|
+------------------------------+------+
|                          NULL| 34480|
|                             0|141453|
|                             1| 35354|
|                             9|  4258|
+------------------------------+------+


IN_BANDA_LARGA


+--------------+------+
|IN_BANDA_LARGA| count|
+--------------+------+
|          NULL| 48774|
|             0| 21576|
|             1|145195|
+--------------+------+



In [41]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation,
    summary
FROM aicat.PRATA.DP01_ESTABELECIMENTOS_ENSINO.snapshots
ORDER BY committed_at DESC
""").show(10, truncate=False)

+-------------------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at       |snapshot_id        |operation|summary                                                                                                                                                                                                                                                                                                                                                                                  